# London cycle hire analysis

Exploration of TfL Santander Cycles journey data for the Data Observatory demo.

## Download the data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from cleaning_lib import (
    drop_false_starts,
    drop_long_hires,
    drop_unknown_stations,
    map_relocated_terminals,
)
from loading_lib import (
    SNAPSHOT_2024_PATH,
    download_file,
    get_stations,
    load_journeys,
    load_stations,
)

RAW_DIR = Path("data/raw")
JOURNEYS_URL = (
    "https://cycling.data.tfl.gov.uk/usage-stats/"
    "399JourneyDataExtract01Jul2024-14Jul2024.csv"
)
BIKEPOINT_URL = "https://api.tfl.gov.uk/BikePoint"

In [2]:
journeys_path = download_file(JOURNEYS_URL, RAW_DIR)
bikepoint_path = download_file(
    BIKEPOINT_URL, RAW_DIR, filename="bikepoint.json"
)
journeys_path, bikepoint_path

(PosixPath('data/raw/399JourneyDataExtract01Jul2024-14Jul2024.csv'),
 PosixPath('data/raw/bikepoint.json'))

## First look

In [3]:
pd.read_csv(journeys_path).head()

,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,141082394,2024-07-15 00:00,1036,"Crawford Street, Marylebone",2024-07-15 00:30,300047,"Alfreda Street, Battersea Park",62221.0,PBSC_EBIKE,30m 20s,1820874
1,141082393,2024-07-14 23:59,1066,"Calshot Street , King's Cross",2024-07-15 00:20,3434,"Sancroft Street, Vauxhall",55323.0,CLASSIC,20m 50s,1250126
2,141082392,2024-07-14 23:59,3449,"Breams Buildings, Holborn",2024-07-15 00:16,200020,"Shadwell Station, Shadwell",20895.0,CLASSIC,17m 2s,1022312
3,141082391,2024-07-14 23:59,1143,"Kensington Church Street, Kensington",2024-07-15 00:03,1116,"Emperor's Gate, South Kensington",57732.0,CLASSIC,4m 16s,256109
4,141082390,2024-07-14 23:59,1007,"Drury Lane, Covent Garden",2024-07-15 00:09,1003,"Chapel Place, Marylebone",56159.0,CLASSIC,10m 30s,630150


In [4]:
pd.read_json(bikepoint_path).head()

,$type,id,url,commonName,placeType,additionalProperties,children,childrenUrls,lat,lon
0,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.529163,-0.109970
1,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_2,/Place/BikePoints_2,"Phillimore Gardens, Kensington",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.499606,-0.197574
2,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_3,/Place/BikePoints_3,"Christopher Street, Liverpool Street",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.521283,-0.084605
3,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_4,/Place/BikePoints_4,"St. Chad's Street, King's Cross",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.530059,-0.120973
4,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_5,/Place/BikePoints_5,"Sedding Street, Sloane Square",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.493130,-0.156876


## Load journeys

`load_journeys` reads a list of journey extracts into one DataFrame. It keeps the columns in `JOURNEY_COLUMNS` under short names, keeps station numbers as strings so their leading zeros match the BikePoint `TerminalName`, and parses the start and end times.

In [5]:
journeys = load_journeys([journeys_path])
journeys.info()

<class 'pandas.DataFrame'>
RangeIndex: 362391 entries, 0 to 362390
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   rental_id       362391 non-null  int64         
 1   bike_id         362390 non-null  Int64         
 2   start           362391 non-null  datetime64[us]
 3   start_terminal  362391 non-null  str           
 4   end             362391 non-null  datetime64[us]
 5   end_terminal    362391 non-null  str           
 6   bike_model      362391 non-null  str           
 7   duration_ms     362391 non-null  int64         
dtypes: Int64(1), datetime64[us](2), int64(2), str(3)
memory usage: 29.2 MB


In [6]:
journeys.head()

,rental_id,bike_id,start,start_terminal,end,end_terminal,bike_model,duration_ms
0,141082394,62221,2024-07-15 00:00:00,001036,2024-07-15 00:30:00,300047,PBSC_EBIKE,1820874
1,141082393,55323,2024-07-14 23:59:00,001066,2024-07-15 00:20:00,003434,CLASSIC,1250126
2,141082392,20895,2024-07-14 23:59:00,003449,2024-07-15 00:16:00,200020,CLASSIC,1022312
3,141082391,57732,2024-07-14 23:59:00,001143,2024-07-15 00:03:00,001116,CLASSIC,256109
4,141082390,56159,2024-07-14 23:59:00,001007,2024-07-15 00:09:00,001003,CLASSIC,630150


## Load stations

`load_stations` reads the saved BikePoint response into one row per docking station. The terminal number and dock count are nested in each station's `additionalProperties` list, so the function unpacks them alongside the name and coordinates.

In [7]:
stations = load_stations(bikepoint_path)
stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   terminal  798 non-null    str    
 1   name      798 non-null    str    
 2   lat       798 non-null    float64
 3   lon       798 non-null    float64
 4   docks     798 non-null    int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 57.0 KB


In [8]:
stations.head()

,terminal,name,lat,lon,docks
0,001023,"River Street , Clerkenwell",51.529163,-0.109970,19
1,001018,"Phillimore Gardens, Kensington",51.499606,-0.197574,37
2,001012,"Christopher Street, Liverpool Street",51.521283,-0.084605,32
3,001013,"St. Chad's Street, King's Cross",51.530059,-0.120973,23
4,003420,"Sedding Street, Sloane Square",51.493130,-0.156876,27


## Exploring for cleaning

### 1. Journeys at unknown stations

`stations` is today's BikePoint list, but the journeys are from July 2024. Which journeys start or end at a station that isn't in the list?

In [9]:
known_terminals = set(stations["terminal"])
start_known = journeys["start_terminal"].isin(known_terminals)
end_known = journeys["end_terminal"].isin(known_terminals)

pd.crosstab(start_known, end_known, margins=True)

end_terminal,False,True,All
start_terminal,,,
False,252,5302,5554
True,5686,351151,356837
All,5938,356453,362391


In [10]:
unknown = ~(start_known & end_known)
print(f"{unknown.sum():,} journeys ({unknown.mean():.1%}) use at "
      "least one unknown station")

11,240 journeys (3.1%) use at least one unknown station


In [11]:
# load_journeys drops station names, so read them from the raw file
# to see what the unknown terminals are
raw_names = pd.read_csv(
    journeys_path,
    usecols=[
        "Start station number",
        "Start station",
        "End station number",
        "End station",
    ],
    dtype=str,
)
station_uses = pd.concat(
    [
        raw_names[["Start station number", "Start station"]]
        .set_axis(["terminal", "name"], axis=1),
        raw_names[["End station number", "End station"]]
        .set_axis(["terminal", "name"], axis=1),
    ]
)

unknown_stations = (
    station_uses[~station_uses["terminal"].isin(known_terminals)]
    .value_counts()
    .rename("journey_ends")
    .reset_index()
)
unknown_stations

,terminal,name,journey_ends
0,022163,"Southwark Station 2, Southwark",1429
1,001020444,"Leonard Circus , Shoreditch_OLD",1406
2,003429444,"Abbey Orchard Street, Westminster_OLD",1208
3,001152,"Marylebone Lane, Marylebone",1106
4,001087,"Belgrave Road, Victoria",1057
5,001056,"Finsbury Square , Moorgate",971
6,001021,"Bouverie Street, Temple",940
7,002635,"Concert Hall Approach 1, South Bank",937
8,300232,"Riverlight South, Nine Elms",876
9,003478,"Guildhouse Street, Victoria",628


In [12]:
# Do the _OLD stations still exist under the same name?
old = unknown_stations[unknown_stations["name"].str.endswith("_OLD")]
current_names = old["name"].str.removesuffix("_OLD").str.strip()
stations[stations["name"].str.strip().isin(current_names)]

,terminal,name,lat,lon,docks
761,003429,"Abbey Orchard Street, Westminster",51.498125,-0.132102,29
762,001020,"Leonard Circus , Shoreditch",51.524696,-0.084439,43


### 2. Which station list?

Station locations only come from the live BikePoint API, which describes the network today. An archived copy from 20 August 2024 (see `reference/README.md`) is much closer to the July 2024 journeys. Comparing the two shows today's list is wrong for some 2024 journeys, not just incomplete.

In [13]:
snapshot_2024 = load_stations(SNAPSHOT_2024_PATH)
print(f"2024 snapshot: {len(snapshot_2024)} stations, "
      f"today's list: {len(stations)} stations")

2024 snapshot: 794 stations, today's list: 798 stations


In [14]:
# Terminals in both lists whose location has changed
both = snapshot_2024.merge(
    stations, on="terminal", suffixes=("_2024", "_today")
)
# Approximate metres per degree at London's latitude
both["moved_m"] = np.hypot(
    (both["lat_2024"] - both["lat_today"]) * 111_000,
    (both["lon_2024"] - both["lon_today"]) * 69_000,
).round()
both.loc[
    both["moved_m"] > 50, ["terminal", "name_2024", "name_today", "moved_m"]
]

,terminal,name_2024,name_today,moved_m
338,002681,"Harriet Street, Knightsbridge","Lowndes Square, Knightsbridge",51.0
340,002639,"King Edward Walk, Waterloo","Brockwell Lido, Brixton",4966.0
374,001058,"Edgware Road Station, Marylebone","Orchard Place, Blackwall Tunnel",12106.0
644,200094,"St. Bride Street, Holborn","St. Bride Street, Holborn",68.0


In [15]:
# The journey file names each station as it was in July 2024, so
# agreement with a list's names shows how well that list fits
journey_names = raw_names[["Start station number", "Start station"]]
journey_names = journey_names.set_axis(["terminal", "name"], axis=1)

for label, candidate in [("2024 snapshot", snapshot_2024),
                         ("today's list", stations)]:
    pairs = journey_names.merge(candidate, on="terminal")
    agree = pairs["name_x"].str.strip() == pairs["name_y"].str.strip()
    print(f"{label}: names agree for {agree.mean():.1%} of journeys")

2024 snapshot: names agree for 99.8% of journeys
today's list: names agree for 99.1% of journeys


We use the 2024 snapshot, filling the few stations it lacks from today's list. `get_stations` records which list each station came from in a `source` column.

In [16]:
stations_2024 = get_stations("2024", RAW_DIR)
stations_2024["source"].value_counts()

source
2024-08-20 snapshot    794
current list            14
Name: count, dtype: int64

## Cleaning

Rules, applied in this order:

1. Map relocated (`_OLD`) terminals, e.g. `001020444`, to their current number `001020`.
2. Drop journeys at stations not in the 2024 list. After step 1 these are only TfL's mechanical workshops.
3. Drop false starts: hires under one minute that end where they started.
4. Drop hires over two hours: we can't know where the bike went, so a route would be meaningless. This removes about 0.6% of hires; 99% of hires take under 91 minutes.

In [17]:
steps = {"loaded": journeys}
steps["relocations mapped"] = map_relocated_terminals(journeys)
steps["unknown stations dropped"] = drop_unknown_stations(
    steps["relocations mapped"], stations_2024
)
steps["false starts dropped"] = drop_false_starts(
    steps["unknown stations dropped"]
)
steps["long hires dropped"] = drop_long_hires(
    steps["false starts dropped"]
)

pd.DataFrame(
    {"journeys": {step: len(df) for step, df in steps.items()}}
).assign(
    removed=lambda d: (d["journeys"].shift() - d["journeys"])
    .fillna(0)
    .astype(int)
)

,journeys,removed
loaded,362391,0
relocations mapped,362391,0
unknown stations dropped,362378,13
false starts dropped,359501,2877
long hires dropped,357229,2272


In [18]:
journeys_clean = steps["long hires dropped"]
print(f"Kept {len(journeys_clean):,} of {len(journeys):,} journeys "
      f"({len(journeys_clean) / len(journeys):.1%})")

Kept 357,229 of 362,391 journeys (98.6%)
